Won't be deployed until I can move away from Render (also why its a jupyter notebook)

Check if valid:
    - Is screenshot of TEC
    - Is on results screen

Check for:
    - Both sides: APM, DPM, SR, Score
    - One side: Time

In [11]:
# imports

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import shutil
import glob
from sklearn.model_selection import train_test_split

In [12]:
# constants
INP_DIR:str = "data/validation"
DATA_DIR:str = "data/validation_processed/training"
TEST_DIR:str = "data/validation_processed/testing"
SEED:int = 136
TRAIN_TEST_SPLIT:float = 0.2

# left top right bottom
YELLOW_FIELD:tuple[float,float,float,float] = (0.0, 0.0, 0.6, 1.0)
BLUE_FIELD:tuple[float,float,float,float] = (0.5, 0.0, 1.0, 1.0)

In [13]:
# create directories

if not os.path.exists(TEST_DIR):
    os.makedirs(TEST_DIR)

if not os.path.exists(f"{TEST_DIR}/0_invalid"):
    os.makedirs(f"{TEST_DIR}/0_invalid")

if not os.path.exists(f"{TEST_DIR}/1_tec_results"):
    os.makedirs(f"{TEST_DIR}/1_tec_results")


if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

if not os.path.exists(f"{DATA_DIR}/0_invalid"):
    os.makedirs(f"{DATA_DIR}/0_invalid")

if not os.path.exists(f"{DATA_DIR}/1_tec_results"):
    os.makedirs(f"{DATA_DIR}/1_tec_results")

In [14]:
# now we need to split the data
files = []
labels = []


# first, index everything 
for class_dir in os.listdir(INP_DIR):
    class_path = f"{INP_DIR}/{class_dir}"

    # makes a list of all the files in the path
    img_files = glob.glob(f"{class_path}/*")

    label:int = 1 if class_dir.startswith("1") else 0

    files.extend(img_files)
    labels.extend([label]*len(img_files))

# split them with train test split

train_val_files, test_files, y, y = train_test_split(
    files,
    labels,
    test_size=TRAIN_TEST_SPLIT,
    random_state=SEED,
    stratify=labels
)

for file_path in train_val_files:
    new_path = file_path.replace("data/validation/", "data/validation_processed/training/")
    shutil.copy(file_path, new_path)

for file_path in test_files:
    new_path = file_path.replace("data/validation/", "data/validation_processed/testing/")
    shutil.copy(file_path, new_path)

In [18]:
# load the data MobileNetV3Small

IMG_SIZE:tuple[int, int] = (224, 224)
BATCH_SIZE:int = 2 # my 6GB of RAM needs limits haha

print("Loading and resizing to", IMG_SIZE)

train_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)


valid_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_tec_results.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_tec_results = train_tec_results.cache().prefetch(buffer_size=AUTOTUNE)
valid_tec_results = valid_tec_results.cache().prefetch(buffer_size=AUTOTUNE)

print(f"\n Found classes: {class_names}")

Loading and resizing to (224, 224)
Found 125 files belonging to 2 classes.


Using 100 files for training.
Found 125 files belonging to 2 classes.
Using 25 files for validation.

 Found classes: ['0_invalid', '1_tec_results']


In [19]:
# Construct a Mobile Net 
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape= IMG_SIZE + (3,),
    alpha=1.0, # determines how much to cut down the model, 1 is fully intact
    minimalistic=False,
    include_top=False, # removes classification layer so we can add our own
    weights="imagenet",
    input_tensor=None,
        classes=1000,
    pooling=None,
    dropout_rate=0.2,
    classifier_activation="softmax",
    include_preprocessing=True,
)

for layer in base_model.layers:
    layer.trainable = False

# binary classification outputs
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
prediciton_layer = tf.keras.layers.Dense(1, activation="sigmoid")

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

processed_input = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
base_model_out = base_model(processed_input, training=False)
pooled_out = global_average_layer(base_model_out)
dropped_out = tf.keras.layers.Dropout(0.2)(pooled_out)

outputs = prediciton_layer(dropped_out)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           577 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 939,697 (3.58 MB)

 Trainable params: 577 (2.25 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [17]:
# load the data but different sizing 


IMG_SIZE:tuple[int, int] = (380, 380)
BATCH_SIZE:int = 1 # my 6GB of RAM needs limits haha


print("Loading and resizing to", IMG_SIZE)

train_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)


valid_tec_results = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_tec_results.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_tec_results = train_tec_results.cache().prefetch(buffer_size=AUTOTUNE)
valid_tec_results = valid_tec_results.cache().prefetch(buffer_size=AUTOTUNE)

print(f"\n Found classes: {class_names}")

Loading and resizing to (380, 380)
Found 125 files belonging to 2 classes.
Using 100 files for training.
Found 125 files belonging to 2 classes.
Using 25 files for validation.

 Found classes: ['0_invalid', '1_tec_results']


In [ ]:
# Construct a EfficientNetB4
base_model = tf.keras.applications.EfficientNetB4(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=IMG_SIZE + (3,),
    pooling=None,
    classes=1000,
    classifier_activation="softmax",
)

for layer in base_model.layers:
    layer.trainable = False

# binary classification outputs
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
prediciton_layer = tf.keras.layers.Dense(1, activation="sigmoid")

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

processed_input = tf.keras.applications.efficientnet.preprocess_input(inputs)
base_model_out = base_model(processed_input, training=False)
pooled_out = global_average_layer(base_model_out)
dropped_out = tf.keras.layers.Dropout(0.2)(pooled_out)

outputs = prediciton_layer(dropped_out)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

model.summary()

2025-11-07 20:25:49.978965: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


71686520/71686520 [==============================] - 17s 0us/step
Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 380, 380, 3)]     0         
                                                                 
 efficientnetb4 (Functional  (None, 12, 12, 1792)      17673823  
 )                                                               
                                                                 
 global_average_pooling2d_3  (None, 1792)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dropout_3 (Dropout)         (None, 1792)              0         
                                                                 
 dense_3 (Dense)             (None, 1)                 1793      
                                                           

In [20]:
print("It's training time!")

checkpointer = tf.keras.callbacks.ModelCheckpoint(
  filepath='tec_check_weights.keras',
  monitor='accuracy',
  verbose=1,
  save_best_only=True
)

It's training time!


In [25]:
model.fit(
    train_tec_results,
    epochs=30,
    validation_data=valid_tec_results,
    callbacks=checkpointer,
    verbose=2
)

Epoch 1/30

Epoch 1: accuracy did not improve from 0.87000
50/50 - 1s - 17ms/step - accuracy: 0.8500 - loss: 0.4592 - val_accuracy: 0.8000 - val_loss: 0.4852
Epoch 2/30

Epoch 2: accuracy did not improve from 0.87000
50/50 - 1s - 18ms/step - accuracy: 0.8100 - loss: 0.4478 - val_accuracy: 0.8000 - val_loss: 0.4824
Epoch 3/30

Epoch 3: accuracy did not improve from 0.87000
50/50 - 1s - 17ms/step - accuracy: 0.8400 - loss: 0.4352 - val_accuracy: 0.8000 - val_loss: 0.4775
Epoch 4/30

Epoch 4: accuracy did not improve from 0.87000
50/50 - 1s - 19ms/step - accuracy: 0.8700 - loss: 0.3966 - val_accuracy: 0.8000 - val_loss: 0.4745
Epoch 5/30

Epoch 5: accuracy did not improve from 0.87000
50/50 - 1s - 17ms/step - accuracy: 0.8500 - loss: 0.4133 - val_accuracy: 0.8000 - val_loss: 0.4716
Epoch 6/30

Epoch 6: accuracy did not improve from 0.87000
50/50 - 1s - 17ms/step - accuracy: 0.8300 - loss: 0.3954 - val_accuracy: 0.8000 - val_loss: 0.4697
Epoch 7/30

Epoch 7: accuracy did not improve from 0

In [40]:
# load testing

test_set = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_paths = test_set.file_paths # this is to see which it misclassified

test_set = test_set.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

loss, acc = model.evaluate(test_set, verbose=2)

print(f"\nFinal Test Accuracy: {acc * 100:.2f}%")

print(test_paths)

Found 32 files belonging to 2 classes.
16/16 - 0s - 16ms/step - accuracy: 0.9375 - loss: 0.2747

Final Test Accuracy: 93.75%
['data/validation_processed/testing/0_invalid/Photos_2v8bg9bVoL.png', 'data/validation_processed/testing/0_invalid/Photos_UIsEqz2BgM.png', 'data/validation_processed/testing/0_invalid/Photos_fLCz7sCP4A.png', 'data/validation_processed/testing/0_invalid/Photos_mVYzPNG7bv.png', 'data/validation_processed/testing/0_invalid/Photos_tdZtlGi3T9.png', 'data/validation_processed/testing/0_invalid/Photos_vS2qTwcTZx.png', 'data/validation_processed/testing/0_invalid/msedge_87Wn6N2S9b.png', 'data/validation_processed/testing/0_invalid/msedge_ExEjTknhTo.png', 'data/validation_processed/testing/0_invalid/msedge_GNAqmjW2jO.png', 'data/validation_processed/testing/0_invalid/msedge_beHJvzRiez.png', 'data/validation_processed/testing/0_invalid/msedge_lIoXhXRrvc.png', 'data/validation_processed/testing/0_invalid/msedge_mD2DHYzlr3.png', 'data/validation_processed/testing/0_invalid/m

In [48]:
# see which ones it failed

predictions = model.predict(test_set)

predicted_labels = np.round(predictions).flatten().astype(int)

true_labels = np.concatenate([y.numpy() for x, y in test_set], axis=0).flatten().astype(int)

misclassified_indices = np.where(predicted_labels != true_labels)[0]

for i in misclassified_indices:
    print(f"\n\npredicted as {predicted_labels[i]} | actually is {true_labels[i]}")
    print(test_paths[i])


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


predicted as 1 | actually is 0
data/validation_processed/testing/0_invalid/steamwebhelper_MQBrK8OoQm.png


predicted as 0 | actually is 1
data/validation_processed/testing/1_tec_results/TetrisEffect-Win64-Shipping_VFEBxeqz6L.jpg


2025-11-10 10:19:18.471961: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
